# 0 Environment and reproducibility

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from dataclasses import dataclass
from functools import lru_cache
from itertools import product
from pathlib import Path
import hashlib
import json
import math
import pickle
import tempfile
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from scipy.special import logsumexp
from sklearn.datasets import load_iris, load_breast_cancer, load_digits, load_diabetes, make_friedman1
from sklearn.metrics import accuracy_score, f1_score, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

MODE = os.getenv("ML441_EXPERIMENT_MODE", "full")
if MODE not in {"full", "smoke"}:
    raise ValueError("ML441_EXPERIMENT_MODE must be 'full' or 'smoke'")
BASE_SEED = DATA_SEED = 441
CODE_VERSION = "ass3-v1"
BUDGET_CU = 3000 if MODE == "full" else 60
H_GRID = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64] if MODE == "full" else [2, 4]
TUNING_SEEDS = tuple(range(10 if MODE == "full" else 2))
FINAL_SEEDS = tuple(range(1000, 1030 if MODE == "full" else 1003))
SGD_GRID = tuple((eta, B) for eta, B in product([0.003, 0.01, 0.03, 0.1, 0.3], [1, 16, 64]))
SCG_GRID = tuple((sigma, lambda_0) for sigma, lambda_0 in product([1e-6, 1e-5, 1e-4], [1e-8, 1e-7, 1e-6]))
LEAPFROG_GRID = tuple((dt0, delta) for dt0, delta in product([0.1, 0.5, 1.0], [0.1, 0.5, 1.0, 5.0]))
OUTPUT_DIR = Path("output")
CACHE_DIR = Path("cache")


In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

# 1 Benchmark problems and preprocessing

The six benchmarks span three classification and three function-approximation tasks. A fixed stratified 20% outer holdout is reserved for final testing. Each seed draws a 75/25 development train/validation split. Input and regression-target scaling are fitted only on the training rows.

In [ ]:
@dataclass(frozen=True)
class Problem:
    id: str
    task: str
    X: np.ndarray
    y: np.ndarray
    description: str


@dataclass(frozen=True)
class PreparedSplit:
    X_train: np.ndarray
    X_val: np.ndarray
    X_test: np.ndarray
    y_train: np.ndarray
    y_val: np.ndarray
    y_test: np.ndarray
    train_idx: np.ndarray
    val_idx: np.ndarray
    test_idx: np.ndarray


PROBLEM_INDEX = {key: i + 1 for i, key in enumerate(("C1", "C2", "C3", "FA1", "FA2", "FA3"))}
PURPOSE_INDEX = {key: i + 1 for i, key in enumerate(("data", "outer", "split", "init", "sgd"))}


def rng_for(problem_id: str, purpose: str, seed: int = 0, H: int = 0) -> np.random.Generator:
    sequence = np.random.SeedSequence(
        [BASE_SEED, PROBLEM_INDEX[problem_id], PURPOSE_INDEX[purpose], int(seed), int(H)]
    )
    return np.random.default_rng(sequence)


@lru_cache(maxsize=1)
def load_problems() -> dict[str, Problem]:
    iris = load_iris()
    cancer = load_breast_cancer()
    digits = load_digits()
    diabetes = load_diabetes(scaled=False)
    z = rng_for("FA1", "data").uniform(-1.0, 1.0, size=500)
    noise = rng_for("FA1", "data", seed=1).normal(0.0, 0.1, size=500)
    X_friedman, y_friedman = make_friedman1(
        n_samples=1000,
        n_features=10,
        noise=1.0,
        random_state=int(rng_for("FA2", "data").integers(0, 2**32 - 1)),
    )
    return {
        "C1": Problem("C1", "classification", iris.data.astype(np.float64), iris.target, "Iris; 3 classes"),
        "C2": Problem("C2", "classification", cancer.data.astype(np.float64), cancer.target, "Breast Cancer Wisconsin Diagnostic; 2 classes"),
        "C3": Problem("C3", "classification", digits.data.astype(np.float64), digits.target, "Optical digits 8×8; 10 classes"),
        "FA1": Problem("FA1", "regression", z[:, None], np.sin(2 * np.pi * z) * np.exp(-z) + noise, "Noisy smooth 1-D function, sigma=0.1"),
        "FA2": Problem("FA2", "regression", X_friedman.astype(np.float64), y_friedman.astype(np.float64), "Friedman #1, sigma=1; 5 irrelevant inputs"),
        "FA3": Problem("FA3", "regression", diabetes.data.astype(np.float64), diabetes.target.astype(np.float64), "Diabetes, raw inputs; real noisy data"),
    }


def make_outer_split(problem: Problem) -> tuple[np.ndarray, np.ndarray]:
    idx = np.arange(len(problem.X))
    stratify = problem.y if problem.task == "classification" else None
    return train_test_split(
        idx,
        test_size=0.2,
        stratify=stratify,
        random_state=int(rng_for(problem.id, "outer").integers(0, 2**32 - 1)),
    )


def make_run_split(problem: Problem, seed: int) -> PreparedSplit:
    dev_idx, test_idx = make_outer_split(problem)
    stratify = problem.y[dev_idx] if problem.task == "classification" else None
    train_idx, val_idx = train_test_split(
        dev_idx,
        test_size=0.25,
        stratify=stratify,
        random_state=int(rng_for(problem.id, "split", seed).integers(0, 2**32 - 1)),
    )
    scaler = StandardScaler().fit(problem.X[train_idx])
    X_train, X_val, X_test = (
        scaler.transform(problem.X[idx]).astype(np.float64)
        for idx in (train_idx, val_idx, test_idx)
    )
    if problem.task == "classification":
        classes = np.unique(problem.y)
        eye = np.eye(len(classes), dtype=np.float64)
        y_train, y_val, y_test = (
            eye[np.searchsorted(classes, problem.y[idx])]
            for idx in (train_idx, val_idx, test_idx)
        )
    else:
        mean = float(np.mean(problem.y[train_idx]))
        scale = float(np.std(problem.y[train_idx])) or 1.0
        y_train, y_val, y_test = (
            ((problem.y[idx] - mean) / scale)[:, None].astype(np.float64)
            for idx in (train_idx, val_idx, test_idx)
        )
    return PreparedSplit(
        X_train, X_val, X_test, y_train, y_val, y_test,
        train_idx, val_idx, test_idx,
    )


In [ ]:
problems = load_problems()
problem_rows = []
for p in problems.values():
    problem_rows.append({
        "problem": p.id, "description": p.description, "task": p.task,
        "n": len(p.X), "inputs": p.X.shape[1],
        "outputs": len(np.unique(p.y)) if p.task == "classification" else 1,
        "class_counts": json.dumps(np.bincount(p.y).tolist()) if p.task == "classification" else "",
        "preprocessing": "training-only input z-score; one-hot targets" if p.task == "classification" else "training-only input and target z-score",
    })
pd.DataFrame(problem_rows).to_csv(OUTPUT_DIR / f"ass3_problems_{MODE}.csv", index=False)

# 2 Network model and gradients

One tanh hidden layer feeds either softmax class probabilities or a linear regression output. Both layers include biases. Weights are initialised uniformly in ±1/√fan-in. The objective is mean cross-entropy or mean squared error.

In [ ]:
@dataclass(frozen=True)
class Shape:
    n_in: int
    H: int
    n_out: int


def pack(W1: np.ndarray, b1: np.ndarray, W2: np.ndarray, b2: np.ndarray) -> np.ndarray:
    return np.concatenate((W1.ravel(), b1.ravel(), W2.ravel(), b2.ravel())).astype(np.float64)


def unpack(w: np.ndarray, shape: Shape) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    n1 = shape.n_in * shape.H
    n2 = n1 + shape.H
    n3 = n2 + shape.H * shape.n_out
    if len(w) != n3 + shape.n_out:
        raise ValueError("weight vector length does not match network shape")
    return (
        w[:n1].reshape(shape.n_in, shape.H),
        w[n1:n2],
        w[n2:n3].reshape(shape.H, shape.n_out),
        w[n3:],
    )


def init_weights(shape: Shape, rng: np.random.Generator) -> np.ndarray:
    first = 1 / math.sqrt(shape.n_in)
    second = 1 / math.sqrt(shape.H)
    return pack(
        rng.uniform(-first, first, (shape.n_in, shape.H)),
        rng.uniform(-first, first, shape.H),
        rng.uniform(-second, second, (shape.H, shape.n_out)),
        rng.uniform(-second, second, shape.n_out),
    )


def _network_values(w: np.ndarray, X: np.ndarray, shape: Shape) -> tuple[np.ndarray, np.ndarray]:
    W1, b1, W2, b2 = unpack(w, shape)
    hidden = np.tanh(X @ W1 + b1)
    logits = hidden @ W2 + b2
    return hidden, logits


def forward(w: np.ndarray, X: np.ndarray, shape: Shape, task: str) -> np.ndarray:
    _, logits = _network_values(w, X, shape)
    if task == "classification":
        return np.exp(logits - logsumexp(logits, axis=1, keepdims=True))
    if task == "regression":
        return logits
    raise ValueError(f"unknown task: {task}")


def loss(w: np.ndarray, X: np.ndarray, y: np.ndarray, shape: Shape, task: str) -> float:
    _, logits = _network_values(w, X, shape)
    if task == "classification":
        return float(np.mean(logsumexp(logits, axis=1) - np.sum(y * logits, axis=1)))
    if task == "regression":
        return float(np.mean((logits - y) ** 2))
    raise ValueError(f"unknown task: {task}")


def loss_grad(w: np.ndarray, X: np.ndarray, y: np.ndarray, shape: Shape, task: str) -> tuple[float, np.ndarray]:
    W1, _, W2, _ = unpack(w, shape)
    hidden, logits = _network_values(w, X, shape)
    if task == "classification":
        log_probs = logits - logsumexp(logits, axis=1, keepdims=True)
        value = -np.mean(np.sum(y * log_probs, axis=1))
        dz2 = (np.exp(log_probs) - y) / len(X)
    elif task == "regression":
        residual = logits - y
        value = np.mean(residual**2)
        dz2 = 2 * residual / residual.size
    else:
        raise ValueError(f"unknown task: {task}")
    dW2 = hidden.T @ dz2
    db2 = dz2.sum(axis=0)
    dz1 = (dz2 @ W2.T) * (1 - hidden**2)
    dW1 = X.T @ dz1
    db1 = dz1.sum(axis=0)
    return float(value), pack(dW1, db1, dW2, db2)


def gradient_relative_error(task: str) -> float:
    shape = Shape(2, 3, 2 if task == "classification" else 1)
    w = init_weights(shape, rng_for("C1", "init", seed=99))
    X = np.array([[-1.0, 0.5], [0.2, -0.3], [1.2, 0.7]])
    y = np.eye(2)[[0, 1, 0]] if task == "classification" else np.array([[0.2], [-0.7], [1.1]])
    _, analytic = loss_grad(w, X, y, shape, task)
    numerical = np.empty_like(w)
    for i in range(len(w)):
        shift = np.zeros_like(w)
        shift[i] = 1e-6
        numerical[i] = (loss(w + shift, X, y, shape, task) - loss(w - shift, X, y, shape, task)) / (2e-6)
    return float(np.linalg.norm(analytic - numerical) / max(1, np.linalg.norm(analytic), np.linalg.norm(numerical)))


In [ ]:
gradient_checks = {task: gradient_relative_error(task) for task in ("classification", "regression")}
assert all(error < 1e-6 for error in gradient_checks.values())
gradient_checks

# 3 Training algorithms

## 3.1 Cost accounting and monitoring

A full error evaluation costs 1 cu, an error-and-gradient evaluation costs 3 cu, and a B-pattern gradient costs 3B/N cu. Validation and training-loss measurements at checkpoints are uncharged. The monitor keeps the lowest-validation-loss weights.

## 3.2 Stochastic gradient descent

Each epoch shuffles training patterns and makes one gradient update per mini-batch, including the shorter last batch.

In [ ]:
class Objective:
    def __init__(self, X: np.ndarray, y: np.ndarray, shape: Shape, task: str):
        self.X = X
        self.y = y
        self.shape = shape
        self.task = task
        self.n = len(X)
        self.cu = 0.0

    def evaluate(self, w: np.ndarray) -> float:
        return loss(w, self.X, self.y, self.shape, self.task)

    def f(self, w: np.ndarray) -> float:
        self.cu += 1.0
        return self.evaluate(w)

    def fg(self, w: np.ndarray) -> tuple[float, np.ndarray]:
        self.cu += 3.0
        return loss_grad(w, self.X, self.y, self.shape, self.task)

    def fg_batch(self, w: np.ndarray, idx: np.ndarray) -> tuple[float, np.ndarray]:
        self.cu += 3.0 * len(idx) / self.n
        return loss_grad(w, self.X[idx], self.y[idx], self.shape, self.task)


class FunctionObjective:
    def __init__(self, f, grad, n: int = 1):
        self._f = f
        self._grad = grad
        self.n = n
        self.cu = 0.0

    def evaluate(self, w: np.ndarray) -> float:
        return float(self._f(w))

    def f(self, w: np.ndarray) -> float:
        self.cu += 1.0
        return self.evaluate(w)

    def fg(self, w: np.ndarray) -> tuple[float, np.ndarray]:
        self.cu += 3.0
        return self.evaluate(w), np.asarray(self._grad(w), dtype=np.float64)

    def fg_batch(self, w: np.ndarray, idx: np.ndarray) -> tuple[float, np.ndarray]:
        self.cu += 3.0 * len(idx) / self.n
        return self.evaluate(w), np.asarray(self._grad(w), dtype=np.float64)


class Monitor:
    def __init__(self, objective, validation_loss, budget_cu: float):
        self.objective = objective
        self.validation_loss = validation_loss
        self.budget_cu = float(budget_cu)
        self.curve = []
        self.best_w = None
        self.best_val_loss = math.inf
        self.best_train_loss = math.inf
        self.cu_at_best = math.nan
        self.diverged = False

    def record(self, w: np.ndarray) -> None:
        train_loss = self.objective.evaluate(w)
        val_loss = float(self.validation_loss(w))
        self.curve.append((float(self.objective.cu), train_loss, val_loss))
        if not (np.isfinite(train_loss) and np.isfinite(val_loss)):
            self.diverged = True
        elif val_loss < self.best_val_loss:
            self.best_w = np.array(w, copy=True)
            self.best_val_loss = val_loss
            self.best_train_loss = train_loss
            self.cu_at_best = float(self.objective.cu)

    @property
    def done(self) -> bool:
        return self.diverged or self.objective.cu >= self.budget_cu


@dataclass(frozen=True)
class OptResult:
    w: np.ndarray
    iterations: int
    scg_restarts_mu: int = 0


def sgd(
    w0: np.ndarray,
    obj: Objective,
    mon: Monitor,
    eta: float,
    batch_size: int,
    rng: np.random.Generator,
) -> OptResult:
    w = np.array(w0, copy=True)
    mon.record(w)
    iterations = 0
    batch_size = min(int(batch_size), obj.n)
    if batch_size < 1 or eta <= 0:
        raise ValueError("SGD requires positive learning rate and batch size")
    while not mon.done:
        for batch in np.array_split(rng.permutation(obj.n), range(batch_size, obj.n, batch_size)):
            _, gradient = obj.fg_batch(w, batch)
            w -= eta * gradient
            if not np.isfinite(w).all():
                break
        iterations += 1
        mon.record(w)
    return OptResult(w, iterations)


## 3.3 Scaled conjugate gradient

The implementation follows Møller (1993): finite-difference directional curvature, adaptive damping, trial-loss acceptance, conjugate-direction updates and periodic restarts. An extra restart handles a non-descent direction on non-quadratic losses.

In [ ]:
def scg(
    w0: np.ndarray,
    obj,
    mon: Monitor,
    sigma: float = 1e-4,
    lambda_0: float = 1e-6,
) -> OptResult:
    if not (0 < sigma <= 1e-4 and 0 < lambda_0 <= 1e-6):
        raise ValueError("SCG requires 0 < sigma <= 1e-4 and 0 < lambda_0 <= 1e-6")
    w = np.array(w0, copy=True, dtype=np.float64)
    mon.record(w)
    f_w, gradient = obj.fg(w)
    r = -gradient
    p = r.copy()
    damping = lambda_0
    damping_bar = 0.0
    success = True
    delta = math.nan
    k = 1
    iterations = 0
    restarts_mu = 0
    while not mon.done and np.linalg.norm(r) > 1e-12:
        p_norm_sq = float(p @ p)
        if p_norm_sq == 0 or not np.isfinite(p_norm_sq):
            mon.diverged = not np.isfinite(p_norm_sq)
            break
        if success:
            sigma_k = sigma / math.sqrt(p_norm_sq)
            _, shifted_gradient = obj.fg(w + sigma_k * p)
            s = (shifted_gradient - gradient) / sigma_k
            delta = float(p @ s)
        delta += (damping - damping_bar) * p_norm_sq
        if delta <= 0:
            damping_bar = 2 * (damping - delta / p_norm_sq)
            delta = -delta + damping * p_norm_sq
            damping = damping_bar
        mu = float(p @ r)
        if mu <= 0:
            p = r.copy()
            success = True
            restarts_mu += 1
            continue
        alpha = mu / delta
        trial_w = w + alpha * p
        trial_f = obj.f(trial_w)
        if not np.isfinite(trial_f):
            mon.record(trial_w)
            iterations += 1
            break
        reduction = 2 * delta * (f_w - trial_f) / (mu * mu)
        if reduction >= 0:
            old_r = r
            w = trial_w
            f_w = trial_f
            _, gradient = obj.fg(w)
            r = -gradient
            damping_bar = 0.0
            success = True
            if k % len(w) == 0:
                p = r.copy()
            else:
                beta = float((r @ r - r @ old_r) / mu)
                p = r + beta * p
            if reduction >= 0.75:
                damping /= 4
        else:
            damping_bar = damping
            success = False
        if reduction < 0.25:
            damping += delta * (1 - reduction) / p_norm_sq
        iterations += 1
        mon.record(w)
        k += 1
    return OptResult(w, iterations, restarts_mu)


## 3.4 LFOP1(b) LeapFrog

The implementation follows Snyman (1983) step numbers literally. It stores the nearby indexed positions, velocities and negative gradients so step 5b can use the previous state and steps 8–9 can jump directly to the next gradient evaluation.

In [ ]:
def leapfrog(
    w0: np.ndarray,
    obj,
    mon: Monitor,
    dt0: float = 0.5,
    delta: float = 1.0,
) -> OptResult:
    if dt0 <= 0 or delta <= 0:
        raise ValueError("LeapFrog requires positive initial time step and maximum step")
    dt = float(dt0)
    x = {0: np.array(w0, copy=True, dtype=np.float64)}
    mon.record(x[0])
    _, gradient = obj.fg(x[0])
    a = {0: -gradient}
    v = {0: 0.5 * a[0] * dt}
    i = 0
    j = 2
    s = 0
    p = 1.0
    k = -1
    iterations = 0
    step_three = True
    while not mon.done:
        if step_three:
            k += 1
            step_norm = float(np.linalg.norm(v[k]) * dt)
            if step_norm < delta:
                p += 0.001
                dt *= p
            else:
                v[k] = delta * v[k] / step_norm
                if s >= 3:
                    dt /= 2
                    x[k] = (x[k] + x[k - 1]) / 2
                    v[k] = (v[k] + v[k - 1]) / 4
                    s = 0
            x[k + 1] = x[k] + v[k] * dt
        if not np.isfinite(x[k + 1]).all():
            mon.diverged = True
            break
        _, gradient = obj.fg(x[k + 1])
        a[k + 1] = -gradient
        v[k + 1] = v[k] + a[k + 1] * dt
        iterations += 1
        mon.record(x[k + 1])
        if float(a[k + 1] @ a[k]) > 0:
            s = 0
        else:
            s += 1
            p = 1.0
        if np.linalg.norm(a[k + 1]) <= 1e-5 or mon.done:
            break
        if np.linalg.norm(v[k + 1]) > np.linalg.norm(v[k]):
            i = 0
            step_three = True
        else:
            x[k + 2] = (x[k + 1] + x[k]) / 2
            i += 1
            if i <= j:
                v[k + 1] = (v[k + 1] + v[k]) / 4
            else:
                v[k + 1] = np.zeros_like(v[k + 1])
                j = 1
            k += 1
            step_three = False
        for old in tuple(x):
            if old < k - 1:
                x.pop(old, None)
                a.pop(old, None)
                v.pop(old, None)
    return OptResult(np.array(x[k + 1] if iterations else x[0], copy=True), iterations)


## 3.5 Optimiser sanity checks

Each check uses a fixed budget independent of experiment mode. The notebook stops before the expensive experiment phases if any check fails.

In [ ]:
def run_sanity_checks() -> pd.DataFrame:
    rows = []

    def add(check: str, measured: float, threshold: float, passed: bool, details: str = "") -> None:
        rows.append({
            "check": check, "measured": float(measured), "threshold": float(threshold),
            "passed": bool(passed), "details": details,
        })

    for task in ("classification", "regression"):
        error = gradient_relative_error(task)
        add(f"gradient_{task}", error, 1e-6, error < 1e-6)

    rng = rng_for("C1", "init", seed=990)
    Q, _ = np.linalg.qr(rng.normal(size=(20, 20)))
    A = Q @ np.diag(np.geomspace(1, 100, 20)) @ Q.T
    b = rng.normal(size=20)
    quadratic = FunctionObjective(
        lambda w: 0.5 * float(w @ A @ w) - float(b @ w),
        lambda w: A @ w - b,
        1,
    )

    class GradientTrace(Monitor):
        def __init__(self, objective, validation_loss, budget_cu):
            super().__init__(objective, validation_loss, budget_cu)
            self.norms = []

        def record(self, w):
            super().record(w)
            self.norms.append(float(np.linalg.norm(A @ w - b)))

    quadratic_monitor = GradientTrace(quadratic, quadratic.evaluate, 600)
    scg(np.zeros(20), quadratic, quadratic_monitor)
    first_hit = next((i for i, norm in enumerate(quadratic_monitor.norms) if norm < 1e-6), math.inf)
    add("scg_quadratic", first_hit, 60, first_hit <= 60, "first iteration with gradient norm < 1e-6")

    def rosenbrock(w):
        return float(100 * (w[1] - w[0] ** 2) ** 2 + (1 - w[0]) ** 2)

    def rosenbrock_grad(w):
        return np.array([
            -400 * w[0] * (w[1] - w[0] ** 2) - 2 * (1 - w[0]),
            200 * (w[1] - w[0] ** 2),
        ])

    rosen = FunctionObjective(rosenbrock, rosenbrock_grad, 1)
    rosen_monitor = Monitor(rosen, rosen.evaluate, 30000)
    rosen_result = leapfrog(np.array([-1.2, 1.0]), rosen, rosen_monitor, 0.5, 1.0)
    distance = float(np.linalg.norm(rosen_monitor.best_w - np.ones(2)))
    add("leapfrog_rosenbrock", distance, 1e-3, distance < 1e-3 and rosen_result.iterations + 1 <= 10000,
        f"{rosen_result.iterations + 1} gradients")

    X = np.arange(5.0)[:, None]
    shape = Shape(1, 2, 1)
    w0 = init_weights(shape, rng_for("FA1", "init", seed=991))
    batch_obj = Objective(X, X.copy(), shape, "regression")
    batch_monitor = Monitor(batch_obj, batch_obj.evaluate, 3)
    sgd(w0, batch_obj, batch_monitor, 0.001, 2, rng_for("FA1", "sgd", seed=991))
    add("sgd_epoch_cost", batch_obj.cu, 3, abs(batch_obj.cu - 3) < 1e-12)

    unit = FunctionObjective(lambda w: 0.5 * float(w @ w), lambda w: w.copy(), 1)
    unit_monitor = Monitor(unit, unit.evaluate, 10)
    scg(np.array([2.0]), unit, unit_monitor)
    add("scg_iteration_cost", unit.cu - 3, 7, abs(unit.cu - 10) < 1e-12)

    iris = load_problems()["C1"]
    split = make_run_split(iris, 0)
    shape = Shape(4, 4, 3)
    w0 = init_weights(shape, rng_for("C1", "init", 0, 4))
    for name in ("sgd", "scg", "leapfrog"):
        objective = Objective(split.X_train, split.y_train, shape, "classification")
        monitor = Monitor(
            objective,
            lambda w: loss(w, split.X_val, split.y_val, shape, "classification"),
            600,
        )
        initial = objective.evaluate(w0)
        if name == "sgd":
            sgd(w0, objective, monitor, 0.1, 16, rng_for("C1", "sgd", 0))
        elif name == "scg":
            scg(w0, objective, monitor)
        else:
            leapfrog(w0, objective, monitor, 0.5, 1.0)
        ratio = monitor.best_train_loss / initial
        add(f"{name}_iris", ratio, 0.5, ratio <= 0.5)

    return pd.DataFrame(rows)


In [ ]:
sanity_checks = run_sanity_checks()
sanity_checks.to_csv(OUTPUT_DIR / f"ass3_sanity_checks_{MODE}.csv", index=False)
assert sanity_checks["passed"].all(), sanity_checks

# 4 Experiment harness

# 5 Phase 1: hidden-unit selection

# 6 Phase 2: control-parameter tuning

# 7 Phase 3: final comparison

# 8 Statistical analysis

# 9 Artifact manifest and completion checks